# 22 匈牙利算法：最小权匹配指派


本课按百科条目写：先定义……调度台在最后，可跳过。

---

## 1. 定义

**指派问题**（assignment problem）：n 个机器人、n 个任务，代价矩阵 \(C\)。\(C_{ij}\) 是第 i 个机器人去做第 j 个任务的代价。求一个一一对应 \(\pi\)（置换 / 完美匹配），使总代价最小：

\[
\min_\pi \sum_{i=0}^{n-1} C_{i,\pi(i)}.
\]

- **输入**：有向图 \(G=(V,E)\)；名单 `robots`；起点字典 `starts[r]`；任务节点列表 `tasks`。通常 \(|robots|=|tasks|=n\)。矩形（车数 ≠ 任务数）时库会补成方阵。
- **输出**：`assign[r] = task`，以及匹配总代价。该代价是各对 **单车最短路之和**，不含多车互相等待。

路径规划里 \(C_{ij}\) **必须**是有向最短路 \(\mathrm{Dijkstra}(s_i, t_j)\)，不是欧氏/曼哈顿。不可达填 \(10^6\)。

**贪心指派**：按名单顺序，每人领剩余任务里当前代价最小者。局部最优，**整体不必最优**。

**匈牙利算法**（Kuhn 1955 / Munkres 1957）：二分图最小权完美匹配的多项式算法。库 `linear_sum_assignment` 用顶标 \(u,v\) 做 Munkres（约 80 行）。本课 **不**把这 80 行当第一份作业：n=2 穷举两种置换 **就是** 最优。

---

## 2. 和谁相邻

| 方法 | 何时用 | 保证 |
|------|--------|------|
| 贪心 | 只要快，或必须按车序立刻定任务 | 无 |
| 匈牙利 / KM | 离线、任务已齐、要最优匹配代价 | 匹配代价最优 |
| 拍卖（23） | 任务一个个到达 | 在线；竞争比可任意差 |
| TPTS（24） | 终身 MAPF：任务流 + 走路 | 启发式 |
| CBS-TA（25） | 匹配之后还要无碰撞路径 | 联合 SOC，不是纯矩阵和 |

匈牙利 **不走路**，只决定谁去哪。几何近 ≠ 有向路近。匹配和也不含等待；SOC 可能让另一匹配更好（25 课）。

---

## 3. 不变量 / 定理

记 \(\mathcal{M}\) 为全部完美匹配。**定理：** 匈牙利的 \(\pi^*\) 满足 \(\sum_i C_{i,\pi^*(i)} \le \sum_i C_{i,\sigma(i)}\) 对任意 \(\sigma\in\mathcal{M}\)。贪心没有这条（第 6 节反例）。

**n=2 推论：** 完美匹配只有两种，穷举取小者 = 匈牙利。这是本课自己写的版本。

**对偶直觉（不必手写 Munkres）：** 顶标 \(u_i,v_j\) 保持 \(C_{ij}-u_i-v_j\ge 0\)。约化为 0 的边组成相等子图；其上的完美匹配即最优（互补松弛）。库数组 `u,v` 就是这对变量。

**补方阵：** n 车 m 任务、\(n\ne m\) 时令 \(k=\max(n,m)\)，空位填 \(+\infty\)（库用 `1e12`）。返回的 (row, col) 丢掉虚拟下标。

---

## 4. 完整伪代码（与下面 Python 一一对应）

```
function CostMatrix(G, robots, starts, tasks):
    for i, r in enumerate(robots):
        for j, t in enumerate(tasks):
            res ← Dijkstra(G, starts[r], t)
            C[i][j] ← res.cost if res.found else 1e6
    return C

function GreedyTwo(robots, tasks, C):
    remaining ← {0,1,...,n-1}
    assign ← {};  total ← 0
    for i, r in enumerate(robots):          # 按名单，不回头
        j* ← argmin_{j in remaining} C[i][j]
        assign[r] ← tasks[j*]
        total ← total + C[i][j*]
        remaining.remove(j*)
    return assign, total

function BruteTwo(robots, tasks, C):        # n=2：两种置换
    π0: assign r0-t0, r1-t1,  cost = C[0][0]+C[1][1]
    π1: assign r0-t1, r1-t0,  cost = C[0][1]+C[1][0]
    return 代价更小者（相等取 π0）
```

库：`linear_sum_assignment(C)` 补方阵 → Munkres → `(rows, cols)`。学生调用，不抄 80 行。

---

## 5. 复杂度

填矩阵 \(n^2\) 次 Dijkstra，通常比匹配更贵。贪心（矩阵已在）\(O(n^2)\)。n=2 穷举 2 次；一般 \(n!\)。KM 是 \(O(n^3)\)，n 是车数不是格子数。

---

## 6. 完整手算 / 小表

课程图 `assign_cross_map`：R0/R1 — J — G0/G1，边权 1 双向。每条 R→J→G 都是 2 步。

|  | G0 | G1 | 说明 |
|--|----|----|------|
| r0@R0 | 2 | 2 | Dijkstra |
| r1@R1 | 2 | 2 | 全相等 |
| 欧氏 r0 | 2.000 | 2.236 | 坐标 (0,1)→(2,1)/(2,0) |
| 欧氏 r1 | 2.236 | 2.000 | 几何近 ≠ 图上近 |

π0：r0–G0 + r1–G1 = 4；π1：r0–G1 + r1–G0 = 4。贪心先 r0 取下标更小的 G0，得到 π0，碰巧最优。

**贪心失败 2×2：** C=[[2,3],[4,10]]。先 R0 取 T0=2，R1 剩 T1=10，合计 **12**。另一置换 3+4=**7**。穷举选 7。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 7. 分步实现（自己写，print traces）

先在真实地图上填 2×2，再实现 `greedy_two` 与 `brute_two`。n=2 的穷举 **就是** 最优匹配。


In [ ]:
from evals.scenarios import assign_pair
from lib.search import dijkstra

g, robots, starts, tasks = assign_pair()
print("robots", robots)
print("starts", starts, "tasks", tasks)

C = []
print("Dijkstra 2x2（有向最短路，不是欧氏）:")
for r in robots:
    row = []
    for t in tasks:
        res = dijkstra(g, starts[r], t, record=False)
        cij = res.cost if res.found else 1e6
        row.append(cij)
        print(f"  {r} {starts[r]} -> {t}  cost={cij}  path={res.path}")
    C.append(row)
print("C =", C)

def greedy_two(robots, tasks, C):
    remaining = list(range(len(tasks)))
    assign, total = {}, 0.0
    for i, r in enumerate(robots):
        best_j = min(remaining, key=lambda j: C[i][j])
        assign[r] = tasks[best_j]
        total += C[i][best_j]
        remaining.remove(best_j)
        print(f"greedy {r} 取 {assign[r]}  本步={C[i][best_j]}  累计={total}  剩余下标={remaining}")
    return assign, total

def brute_two(robots, tasks, C):
    # n=2 只有两种置换；这就是最优，不要在这里抄 80 行 Munkres
    cand = [
        ({robots[0]: tasks[0], robots[1]: tasks[1]}, C[0][0] + C[1][1]),
        ({robots[0]: tasks[1], robots[1]: tasks[0]}, C[0][1] + C[1][0]),
    ]
    for a, cost in cand:
        print("brute 匹配", a, "代价", cost)
    return min(cand, key=lambda x: x[1])

print("--- greedy_two ---")
print("结果", greedy_two(robots, tasks, C))
print("--- brute_two ---")
print("结果", brute_two(robots, tasks, C))


## 8. 逐行实现表

| 行 | 作用 |
|----|------|
| `dijkstra(..., record=False)` | 只要 cost/path |
| `found else 1e6` | 不可达写大数，避免 inf 加法 |
| `remaining` 列下标 | 当前行在剩余列上 `min` |
| `brute_two` 两条 | π0 对角线、π1 反对角线；相等取 π0 |

下面用贪心失败的数字矩阵对照库。


In [ ]:
from lib.algos.assignment import linear_sum_assignment, hungarian_assign, greedy_assign, cost_matrix

C_bad = [
    [2.0, 3.0],
    [4.0, 10.0],
]
rb, tb = ["R0", "R1"], ["T0", "T1"]
print("C_bad 贪心", greedy_two(rb, tb, C_bad))
print("C_bad 穷举", brute_two(rb, tb, C_bad))
rows, cols = linear_sum_assignment(C_bad)
print("库 KM rows, cols", rows, cols)
print("库 KM 匹配", {rb[i]: tb[j] for i, j in zip(rows, cols)})
print("库 KM 代价", sum(C_bad[i][j] for i, j in zip(rows, cols)))

print("--- 对照真实地图上的库函数 ---")
print("greedy_assign", greedy_assign(g, robots, starts, tasks))
hung = hungarian_assign(g, robots, starts, tasks)
print("hungarian_assign", hung["assign"], "cost", hung["cost"])
print("Dijkstra 矩阵", hung["matrix"])
print("欧氏矩阵  ", hung["euclidean"])
print("used_topo", hung["used_topo"])
print("cost_matrix topo", cost_matrix(g, robots, starts, tasks, use_topo=True))
print("cost_matrix 启发", cost_matrix(g, robots, starts, tasks, use_topo=False))


## 9. 常见 bug

1. **用欧氏 / 曼哈顿填 C。** 本图欧氏不是全 2；有向路网更会错。必须 Dijkstra。
2. **矩形不补方阵。** 2 车 3 任务直接跑方阵 KM 会越界或丢任务。库先 pad 再解。
3. **把匹配代价当成 SOC。** 匹配和 = 各走各的最短路。CBS 在 J 口等待后 SOC 可以是 5，不是 4（25 课）。
4. **不可达写成 inf 又拿去加。** `inf+3` 仍是 inf，排序也别扭。约定 `1e6`。
5. **贪心按任务循环而不是按车。** 与本课伪代码不一致，反例数字会对不上。

## 10. 对照库

`cost_matrix` / `greedy_assign` / `linear_sum_assignment` / `hungarian_assign` 都在 `lib/algos/assignment.py`。KM 补方阵并返回下标；`used_topo=True` 表示 C 来自 Dijkstra。eval `test_22`：匈牙利代价 ≤ 贪心。本图两者都是 4。


In [ ]:
from lib.studio.widget import PlannerStudio

studio = PlannerStudio()
studio.map = g.to_studio()
studio.algo = "assignment"
studio


## 条目小结

| 项目 | 内容 |
|------|------|
| 问题 | 二分图最小权完美匹配 |
| \(C_{ij}\) | **有向** Dijkstra，不是欧氏 |
| 贪心 | 按车序领剩余最小；不必最优 |
| n=2 | 两种置换穷举 = 最优 |
| 匈牙利 | Kuhn 1955 / Munkres 1957，\(O(n^3)\) |
| 不保证 | 含等待的 SOC 最优（那是 CBS-TA） |

## 练习

1. 纸上跑 `C_bad=[[2,3],[4,10]]`：写出贪心步骤与两种置换。最优是哪一个？
2. 若 `C[0][1]` 不可达改成 `1e6`，穷举会选哪对？把 `1e6` 错写成 `0` 会怎样？
3. 证明：n=2 时比较两条对角线（两种置换）就是最小权完美匹配。n=3 要比较几条？
